# 01 — Generate the complete paper trajectory database from scratch

This notebook generates every `.npz` trajectory used by the paper. The directory is shipped without those files.

**Cost warning.** The full campaign contains 114 independent realizations, including 24 `K_max=192`, `768×768`-grid runs with 1000 tracers. Runtime is strongly machine-dependent and can be many hours or longer on a laptop. The notebook is restartable: existing files are skipped unless `FORCE=True`.

The roughness ensemble preserves the exact production provenance. Some roughness seeds were generated in the initial shorter campaign and others in the extended campaign; the configuration file records these groups separately while writing the same final filename pattern.

In [1]:
from pathlib import Path
import os, sys

def find_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "scripts").is_dir():
            return candidate
    raise RuntimeError("Repository root not found. Start Jupyter inside the extracted directory.")

ROOT = find_root(Path.cwd())
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print("ROOT =", ROOT)


ROOT = /mnt/data/work_release_v3/LagrangianEllipsoid_Corrected_Release_v3/2_github_repository


In [2]:
import json
import pandas as pd

CONFIG_PATH = ROOT / "configs" / "ensembles.json"
configs = json.loads(CONFIG_PATH.read_text())
rows=[]
for name,c in configs.items():
    rows.append({
        "ensemble":name, "mode":c["mode"], "n_seeds":len(c["seeds"]),
        "zeta":c["zeta"], "Kmax":c["kmax"], "ngrid":c["ngrid"],
        "T":c["T"], "N":c["N"], "outdir":c["outdir"]
    })
df=pd.DataFrame(rows)
display(df)
print("Total realizations:", int(df.n_seeds.sum()))

,ensemble,mode,n_seeds,zeta,Kmax,ngrid,T,N,outdir
0,decisive_k192,full,24,0.333333,192,768,20.0,1000,data/raw/decisive_k192
1,roughness_zeta_033,full,12,0.333333,128,512,9.1,1000,data/raw/roughness_k128
2,roughness_zeta_025_short,full,6,0.250000,128,512,9.1,1000,data/raw/roughness_k128
3,roughness_zeta_025_extended,full,6,0.250000,128,512,15.0,1000,data/raw/roughness_k128
4,roughness_zeta_050_short,full,6,0.500000,128,512,9.1,1000,data/raw/roughness_k128
5,roughness_zeta_050_extended,full,4,0.500000,128,512,15.0,1000,data/raw/roughness_k128
6,roughness_zeta_067_short,full,6,0.666667,128,512,9.1,1000,data/raw/roughness_k128
7,roughness_zeta_067_extended,full,4,0.666667,128,512,15.0,1000,data/raw/roughness_k128
8,control_k64,shape,8,0.333333,64,256,20.0,1000,data/raw/controls_k64
9,control_r0_half,shape,8,0.333333,128,512,15.0,1000,data/raw/controls_r0


Total realizations: 114


In [3]:
# Execution controls
WORKERS = 1        # Use 1 first. Increase cautiously; each Kmax=192 worker is memory intensive.
FORCE = False      # False makes the workflow resumable.

# Exact order used in the production campaign.
ENSEMBLE_ORDER = [
    "decisive_k192",
    "roughness_zeta_033",
    "roughness_zeta_025_short", "roughness_zeta_025_extended",
    "roughness_zeta_050_short", "roughness_zeta_050_extended",
    "roughness_zeta_067_short", "roughness_zeta_067_extended",
    "control_k64", "control_r0_half", "control_r0_double",
    "control_memory_constant", "control_memory_short", "control_nested_N",
]
assert set(ENSEMBLE_ORDER) == set(configs)

In [4]:
import subprocess, time, os
from datetime import datetime, timezone

logdir = ROOT / "logs"
logdir.mkdir(exist_ok=True)
env = os.environ.copy()
# Avoid BLAS oversubscription if WORKERS is larger than one.
env.setdefault("OMP_NUM_THREADS", "1")
env.setdefault("OPENBLAS_NUM_THREADS", "1")
env.setdefault("MKL_NUM_THREADS", "1")

start_all=time.time()
for ensemble in ENSEMBLE_ORDER:
    print("\n" + "="*80)
    print("Starting", ensemble, datetime.now(timezone.utc).isoformat())
    cmd=[sys.executable, str(ROOT/"scripts"/"run_ensemble.py"), ensemble, "--workers", str(WORKERS)]
    if FORCE:
        cmd.append("--force")
    t0=time.time()
    with (logdir/f"{ensemble}.log").open("a", encoding="utf-8") as log:
        log.write("\nCOMMAND: " + " ".join(cmd) + "\n")
        proc=subprocess.run(cmd, cwd=ROOT, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        log.write(proc.stdout)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        raise RuntimeError(f"{ensemble} failed; inspect logs/{ensemble}.log")
    print(f"Finished {ensemble} in {(time.time()-t0)/60:.1f} min")
print(f"All requested ensembles completed in {(time.time()-start_all)/3600:.2f} h")


Starting decisive_k192 2026-07-20T13:13:22.393605+00:00


skip rough13_k192_seed00.npz
skip rough13_k192_seed01.npz
skip rough13_k192_seed02.npz
skip rough13_k192_seed03.npz
skip rough13_k192_seed04.npz
skip rough13_k192_seed05.npz
skip rough13_k192_seed06.npz
skip rough13_k192_seed07.npz
skip rough13_k192_seed08.npz
skip rough13_k192_seed09.npz
skip rough13_k192_seed10.npz
skip rough13_k192_seed11.npz
skip rough13_k192_seed12.npz
skip rough13_k192_seed13.npz
skip rough13_k192_seed14.npz
skip rough13_k192_seed15.npz
skip rough13_k192_seed16.npz
skip rough13_k192_seed17.npz
skip rough13_k192_seed18.npz
skip rough13_k192_seed19.npz
skip rough13_k192_seed20.npz
skip rough13_k192_seed21.npz
skip rough13_k192_seed22.npz
skip rough13_k192_seed23.npz

Finished decisive_k192 in 0.0 min

Starting roughness_zeta_033 2026-07-20T13:13:23.319419+00:00


skip rough13_k128_seed00.npz
skip rough13_k128_seed01.npz
skip rough13_k128_seed02.npz
skip rough13_k128_seed03.npz
skip rough13_k128_seed04.npz
skip rough13_k128_seed05.npz
skip rough13_k128_seed06.npz
skip rough13_k128_seed07.npz
skip rough13_k128_seed08.npz
skip rough13_k128_seed09.npz
skip rough13_k128_seed10.npz
skip rough13_k128_seed11.npz

Finished roughness_zeta_033 in 0.0 min

Starting roughness_zeta_025_short 2026-07-20T13:13:24.407498+00:00


skip rough025_k128_seed00.npz
skip rough025_k128_seed01.npz
skip rough025_k128_seed02.npz
skip rough025_k128_seed03.npz
skip rough025_k128_seed04.npz
skip rough025_k128_seed05.npz

Finished roughness_zeta_025_short in 0.0 min

Starting roughness_zeta_025_extended 2026-07-20T13:13:25.515579+00:00


skip rough025_k128_seed06.npz
skip rough025_k128_seed07.npz
skip rough025_k128_seed08.npz
skip rough025_k128_seed09.npz
skip rough025_k128_seed10.npz
skip rough025_k128_seed11.npz

Finished roughness_zeta_025_extended in 0.0 min

Starting roughness_zeta_050_short 2026-07-20T13:13:26.721653+00:00


...

skip rough13_k128_r0half_seed00.npz
skip rough13_k128_r0half_seed01.npz
skip rough13_k128_r0half_seed02.npz
skip rough13_k128_r0half_seed03.npz
skip rough13_k128_r0half_seed04.npz
skip rough13_k128_r0half_seed05.npz
skip rough13_k128_r0half_seed06.npz
skip rough13_k128_r0half_seed07.npz

Finished control_r0_half in 0.0 min

Starting control_r0_double 2026-07-20T13:13:32.716203+00:00


skip rough13_k128_r0double_seed00.npz
skip rough13_k128_r0double_seed01.npz
skip rough13_k128_r0double_seed02.npz
skip rough13_k128_r0double_seed03.npz
skip rough13_k128_r0double_seed04.npz
skip rough13_k128_r0double_seed05.npz
skip rough13_k128_r0double_seed06.npz
skip rough13_k128_r0double_seed07.npz

Finished control_r0_double in 0.0 min

Starting control_memory_constant 2026-07-20T13:13:33.948650+00:00


skip rough13_k128_memconst_seed00.npz
skip rough13_k128_memconst_seed01.npz
skip rough13_k128_memconst_seed02.npz
skip rough13_k128_memconst_seed03.npz
skip rough13_k128_memconst_seed04.npz
skip rough13_k128_memconst_seed05.npz
skip rough13_k128_memconst_seed06.npz
skip rough13_k128_memconst_seed07.npz

Finished control_memory_constant in 0.0 min

Starting control_memory_short 2026-07-20T13:13:34.916387+00:00


skip rough13_k128_memshort_seed00.npz
skip rough13_k128_memshort_seed01.npz
skip rough13_k128_memshort_seed02.npz
skip rough13_k128_memshort_seed03.npz
skip rough13_k128_memshort_seed04.npz
skip rough13_k128_memshort_seed05.npz
skip rough13_k128_memshort_seed06.npz
skip rough13_k128_memshort_seed07.npz

Finished control_memory_short in 0.0 min

Starting control_nested_N 2026-07-20T13:13:35.829152+00:00


skip rough13_k128_nestedN_seed00.npz
skip rough13_k128_nestedN_seed01.npz
skip rough13_k128_nestedN_seed02.npz
skip rough13_k128_nestedN_seed03.npz
skip rough13_k128_nestedN_seed04.npz
skip rough13_k128_nestedN_seed05.npz

Finished control_nested_N in 0.0 min
All requested ensembles completed in 0.00 h


In [5]:
from collections import Counter
import hashlib
from datetime import datetime, timezone

expected={
 "decisive_k192":24, "roughness_k128":44, "controls_k64":8,
 "controls_r0":16, "controls_memory":16, "controls_N":6,
}
actual={name:len(list((ROOT/"data"/"raw"/name).glob("*.npz"))) for name in expected}
print("Inventory:", actual)
assert actual == expected, (actual, expected)

manifest={
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "config_sha256":hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest(),
 "inventory":actual,
 "workers":WORKERS,
}
manifest_path=ROOT/"data"/"raw"/"generation_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print("Wrote", manifest_path.relative_to(ROOT))

Inventory: {'decisive_k192': 24, 'roughness_k128': 44, 'controls_k64': 8, 'controls_r0': 16, 'controls_memory': 16, 'controls_N': 6}
Wrote data/raw/generation_manifest.json
